# 07.3 — Estudos Causais AIPW nos Recém-Nascidos

**Purpose:** AIPW causal inference with 5-fold cross-fitting, covariate balance
diagnostics (SMD), E-value sensitivity analysis, and a forest plot.
Loads shared artifacts produced by `07.1_carga_rn.py`.

## 1. Setup

In [ ]:
from __future__ import annotations
import sys, warnings, pickle, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier as GBC
from sklearn.model_selection import KFold
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        "fp_xlabel":  "ATE (pp)",
        "fp_title":   "Forest Plot — Fatores de Risco Neonatais (AIPW 5-fold CF)",
    },
    "en": {
        "fp_xlabel":  "ATE (pp)",
        "fp_title":   "Forest Plot — Neonatal Risk Factors (AIPW 5-fold CF)",
    },
}
T = _T[LANG]

ROOT    = Path(__file__).resolve().parent.parent
ART_DIR = ROOT / "out" / "artifacts_rn"
OUT_DIR = ROOT / "out" / "07.3_causal_rn"
PLOT_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

print(f"ROOT    : {ROOT}")
print(f"ART_DIR : {ART_DIR}")

## Load artifacts

In [ ]:
with open(ART_DIR / "rn_dataset.pkl", "rb") as f:
    _art = pickle.load(f)

df = _art["df"]
print(f"df : {len(df):,} bebes")

## 2. Preprocessor factory

In [ ]:
def _pre(X: pd.DataFrame) -> ColumnTransformer:
    """ColumnTransformer with MICE for numeric and OHE for categorical."""
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    return ColumnTransformer(
        [
            (
                "num",
                Pipeline([
                    ("i", IterativeImputer(max_iter=10, random_state=42)),
                    ("s", StandardScaler()),
                ]),
                num,
            ),
            (
                "cat",
                Pipeline([
                    ("i", SimpleImputer(strategy="most_frequent")),
                    ("o", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                cat,
            ),
        ],
        remainder="drop",
    )

## 3. AIPW with 5-fold cross-fitting

In [ ]:
def _aipw_cf(df_a: pd.DataFrame, T_col: str, Y_col: str, conf: list, n_boot: int = 500):
    """AIPW with 5-fold cross-fitting.

    Returns a dict with ate_pp, IC95%, p-value, n, prevalences; or None if
    insufficient overlap/sample.
    """
    CONF = [c for c in conf if c in df_a.columns]
    sub  = df_a[[T_col, Y_col] + CONF].dropna(subset=[T_col, Y_col])
    sub  = sub[sub[Y_col].isin([0, 1])]

    if sub[T_col].sum() < 20 or (sub[T_col] == 0).sum() < 20:
        return None

    Tv = sub[T_col].values.astype(int)
    Yv = sub[Y_col].values.astype(int)
    pre = _pre(sub[CONF])
    X   = pre.fit_transform(sub[CONF])

    kf  = KFold(n_splits=5, shuffle=True, random_state=42)
    s1  = np.zeros(len(Yv))
    s0  = np.zeros(len(Yv))

    for tr_idx, te_idx in kf.split(X):
        ps_f = GBC(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
        ps_f.fit(X[tr_idx], Tv[tr_idx])
        ps_te = np.clip(ps_f.predict_proba(X[te_idx])[:, 1], 0.05, 0.95)

        X_out = np.hstack([X[tr_idx], Tv[tr_idx].reshape(-1, 1)])
        om_f  = GBC(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
        om_f.fit(X_out, Yv[tr_idx])

        mu1_te = om_f.predict_proba(
            np.hstack([X[te_idx], np.ones((len(te_idx), 1))])
        )[:, 1]
        mu0_te = om_f.predict_proba(
            np.hstack([X[te_idx], np.zeros((len(te_idx), 1))])
        )[:, 1]

        s1[te_idx] = mu1_te + Tv[te_idx] * (Yv[te_idx] - mu1_te) / (ps_te + 1e-9)
        s0[te_idx] = mu0_te + (1 - Tv[te_idx]) * (Yv[te_idx] - mu0_te) / (1 - ps_te + 1e-9)

    ate = (s1 - s0).mean()
    rng = np.random.default_rng(42)
    b   = [
        (
            s1[rng.integers(0, len(Yv), len(Yv))]
            - s0[rng.integers(0, len(Yv), len(Yv))]
        ).mean()
        for _ in range(n_boot)
    ]
    se      = np.std(b)
    lo, hi  = np.percentile(b, [2.5, 97.5])
    pval    = 2 * (1 - scipy_stats.norm.cdf(abs(ate / se))) if se > 0 else 1.0

    return {
        "n":         len(sub),
        "n_exp":     int(Tv.sum()),
        "prev_ctrl": round(Yv[Tv == 0].mean() * 100, 2),
        "prev_exp":  round(Yv[Tv == 1].mean() * 100, 2),
        "ate_pp":    round(ate * 100, 4),
        "ic_lo":     round(lo  * 100, 4),
        "ic_hi":     round(hi  * 100, 4),
        "se":        round(se,         4),
        "p":         round(pval,        4),
    }

## 4. E-value (sensitivity to unmeasured confounding)

In [ ]:
def e_value_rd(ate_pp: float, baseline_pct: float) -> float:
    """E-value for a risk difference on the risk-ratio scale."""
    p0 = baseline_pct / 100
    p1 = p0 + ate_pp / 100
    if p1 <= 0 or p0 <= 0:
        return np.nan
    rr = max(p1 / p0, p0 / p1)
    return round(rr + np.sqrt(rr * (rr - 1)), 3)

## 5. Standardized Mean Difference (balance diagnostic)

In [ ]:
def _smd(df: pd.DataFrame, col: str, T_col: str) -> float:
    """Pooled SMD for a single covariate."""
    g1 = pd.to_numeric(df[df[T_col] == 1][col], errors="coerce").dropna()
    g0 = pd.to_numeric(df[df[T_col] == 0][col], errors="coerce").dropna()
    if len(g1) == 0 or len(g0) == 0:
        return np.nan
    pool = np.sqrt((g1.std() ** 2 + g0.std() ** 2) / 2)
    return abs(g1.mean() - g0.mean()) / pool if pool > 0 else 0.0

## 6. Confounders

In [ ]:
# Prefer lowercase; add uppercase fallbacks if lowercase not present
CONF = [
    c for c in [
        "idade_mae", "segmento_mae", "cesario", "escore_triagem_mae",
        "g_obs", "a_obs", "risco_gestacao", "gestacao_gemelar",
    ]
    if c in df.columns
]
# Uppercase fallback for enriched columns
CONF += [c for c in ["G_obs", "A_obs"] if c in df.columns and c.lower() not in CONF]
CONF = list(dict.fromkeys(CONF))  # deduplicate, preserve order

print(f"Confounders ({len(CONF)}): {CONF}")

# Flag PE
df["T_pe"] = df["cid_mae"].fillna("").str.match(r"^O1[0-6]").astype(int)

# Macrossomia binary (for C7)
df["macrossomia_bin"] = (df["peso_g"] >= 4000).astype(float)

# Flag aborto (A_obs or a_obs >= 1)
_a_col = "A_obs" if "A_obs" in df.columns else ("a_obs" if "a_obs" in df.columns else None)
if _a_col:
    df["T_aborto"] = (pd.to_numeric(df[_a_col], errors="coerce").fillna(0) >= 1).astype(int)
else:
    df["T_aborto"] = 0

# Flag grande multipara (G_obs >= 5)
_g_col = "G_obs" if "G_obs" in df.columns else ("g_obs" if "g_obs" in df.columns else None)
if _g_col:
    df["T_multipara"] = (pd.to_numeric(df[_g_col], errors="coerce").fillna(0) >= 5).astype(int)
else:
    df["T_multipara"] = 0

# Flag mae diabetica O24
df["T_diabetes"] = df["cid_mae"].fillna("").str.startswith("O24").astype(int)

## 7. Causal studies

In [ ]:
estudos: list[dict] = []
dc = df[df["vinculado_mae"] == 1].copy()

print("=" * 60)
print("ESTUDOS CAUSAIS — RN (AIPW 5-fold cross-fitting, n_boot=500)")
print("=" * 60)

_balance_cols = ["idade_mae", "cesario", "escore_triagem_mae"]

for titulo, T_col, Y_col, desc in [
    ("C1 - Prematuridade <37s -> Obito",       "prematuro",     "obito_rn",       "prematuro vs a termo"),
    ("C2 - Muito Prematuro <32s -> Obito",     "mt_prematuro",  "obito_rn",       "<32s vs >=32s"),
    ("C3 - Baixo Peso <2.500g -> Obito",       "baixo_peso",    "obito_rn",       "baixo peso vs adequado"),
    ("C4 - Mae PE/HAS -> Obito RN",            "T_pe",          "obito_rn",       "mae O10-O16 vs outras"),
    ("C5 - Aborto materno (A>=1) -> Apgar dep","T_aborto",      "apgar_dep",      "A>=1 vs A=0"),
    ("C6 - Grande multipara (G>=5) -> Obito",  "T_multipara",   "obito_rn",       "G>=5 vs G<5"),
    ("C7 - Mae diabetica (O24) -> Macrossomia","T_diabetes",    "macrossomia_bin","O24 vs outras"),
]:
    if T_col not in dc.columns:
        print(f"\n  {titulo}: SKIP (coluna {T_col} ausente)")
        continue

    r = _aipw_cf(dc, T_col, Y_col, CONF, n_boot=500)
    if r is None:
        print(f"\n  {titulo}: SKIP (amostra insuficiente)")
        continue

    # Balance report
    print(f"\n  {titulo}  [{desc}]")
    print(f"  N={r['n']:,}  T=1:{r['n_exp']:,}  ctrl:{r['prev_ctrl']}%  exp:{r['prev_exp']}%")
    sig = "***" if r["p"] < 0.001 else "**" if r["p"] < 0.01 else "*" if r["p"] < 0.05 else "ns"
    print(f"  ATE={r['ate_pp']:+.2f}pp  IC95%[{r['ic_lo']:+.2f},{r['ic_hi']:+.2f}]  "
          f"p={r['p']:.4f} {sig}")

    _bal = {c: round(_smd(dc, c, T_col), 3) for c in _balance_cols if c in dc.columns}
    if _bal:
        print(f"  Balance SMD: " + "  ".join(f"{k}={v}" for k, v in _bal.items()))

    ev = e_value_rd(r["ate_pp"], r["prev_ctrl"])
    print(f"  E-value: {ev}")

    estudos.append({
        "estudo":      titulo,
        "T_col":       T_col,
        "Y_col":       Y_col,
        "e_value":     ev,
        **r,
    })

## 8. Forest Plot with E-value annotation

In [ ]:
if estudos:
    est = pd.DataFrame(estudos)
    n_est = len(est)
    fig_fp, ax_fp = plt.subplots(figsize=(13, max(5, n_est * 1.4)))

    for i, row in est.iterrows():
        cor = "#d62728" if row["ate_pp"] > 0 else "#2ca02c"
        ax_fp.errorbar(
            row["ate_pp"], i,
            xerr=[[row["ate_pp"] - row["ic_lo"]], [row["ic_hi"] - row["ate_pp"]]],
            fmt="o", color=cor, capsize=5, markersize=8
        )
        sig = "***" if row["p"] < 0.001 else "**" if row["p"] < 0.01 else "*" if row["p"] < 0.05 else "ns"
        ev_str = f"  E={row['e_value']}" if not pd.isna(row["e_value"]) else ""
        ax_fp.text(
            row["ic_hi"] + 0.2, i,
            f"p={row['p']:.3f} {sig}{ev_str}",
            va="center", fontsize=8.5
        )

    ax_fp.axvline(0, color="black", lw=0.8, ls="--")
    ax_fp.set_yticks(range(n_est))
    ax_fp.set_yticklabels(est["estudo"], fontsize=9)
    ax_fp.set_xlabel(T["fp_xlabel"])
    ax_fp.set_title(T["fp_title"])
    ax_fp.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "forest_plot_rn.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"\n[07.3] forest_plot_rn.png salvo.")

## 9. Save results

In [ ]:
if estudos:
    est_df = pd.DataFrame(estudos)
    est_df.to_csv(OUT_DIR / "estudos_causais_rn.csv", index=False)

    # JSON summary
    resumo = [
        {
            "estudo":   r["estudo"],
            "ate_pp":   r["ate_pp"],
            "ic_lo":    r["ic_lo"],
            "ic_hi":    r["ic_hi"],
            "p":        r["p"],
            "e_value":  r["e_value"],
        }
        for r in estudos
    ]
    with open(OUT_DIR / "resumo_causal_rn.json", "w", encoding="utf-8") as f:
        json.dump(resumo, f, indent=2, ensure_ascii=False, default=str)

    print(f"[07.3] estudos_causais_rn.csv + resumo_causal_rn.json salvos.")

In [ ]:
print("\n" + "="*60)
print("  07 RN - Resumo Causal")
print("="*60)
if estudos:
    _hdr = f"  {'Estudo':<42} {'ATE':>7}  {'IC95%':^18}  {'p':>7}  {'E-val':>6}"
    print(_hdr)
    print("  " + "-" * (len(_hdr) - 2))
    for r in estudos:
        sig = "***" if r["p"] < 0.001 else "**" if r["p"] < 0.01 else "*" if r["p"] < 0.05 else "ns "
        ev  = f"{r['e_value']:.2f}" if not pd.isna(r.get("e_value", np.nan)) else "  —  "
        print(
            f"  {r['estudo']:<42} {r['ate_pp']:>+7.2f}  "
            f"[{r['ic_lo']:>+6.2f},{r['ic_hi']:>+6.2f}]  "
            f"{r['p']:>7.4f}{sig}  {ev:>6}"
        )
else:
    print("  Nenhum estudo concluido.")
print("="*60)
print(f"  Resultados em : {OUT_DIR}")
print("="*60)